In [1]:
import torch
from torch.utils.data import DataLoader, ConcatDataset
import torch.nn as nn
import os
import random
from torchinfo import summary
import pickle
import optuna

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print('The model is running on:', DEVICE) 

LABEL_PADDING_VALUE = 99
FEATURE_PADDING_VALUE = 0
NUM_FEATURES = 10
NUM_CLASSES = 4
BATCH_SIZE = 32

The model is running on: cuda:0


In [5]:
#########################################################################################################
with open("/home/haidiri/Desktop/andi/data_from_andi/new_data/max_200_train_instances.pkl", "rb") as file:
    train_data = pickle.load(file)

with open("/home/haidiri/Desktop/andi/data_from_andi/new_data/max_200_train_instances_with_state3bias.pkl", "rb") as file:
    train_data_state3 = pickle.load(file)

train_data.extend(train_data_state3)

##########################################################################################################

with open("/home/haidiri/Desktop/andi/data_from_andi/new_data/max_200_val_instances.pkl", "rb") as file:
    val_data = pickle.load(file)

with open("/home/haidiri/Desktop/andi/data_from_andi/new_data/max_200_val_instances_with_state3bias.pkl", "rb") as file:
    val_data_state_3 = pickle.load(file)

val_data.extend(val_data_state_3)

random.shuffle(train_data)
random.shuffle(val_data)

move_to_val = 0.2 * len(train_data)

val_data.extend(train_data[:int(move_to_val)])
train_data = train_data[int(move_to_val):]

print("Train data: ", len(train_data))
print("Val data: ", len(val_data))

conc_train = ConcatDataset(train_data)
conc_val = ConcatDataset(val_data)

training_loader = DataLoader(conc_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_batch)
val_loader = DataLoader(conc_val, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_batch)

print("Data", len(training_loader), len(val_loader))


Train data:  1633896
Val data:  663769
Data 51060 20743


In [ ]:
model_summary = summary(Seq2Seq(num_inputs=NUM_FEATURES).to(DEVICE), input_size=(BATCH_SIZE, 500, NUM_FEATURES))
print(model_summary)

continuous_loss_fn = nn.L1Loss(reduction='none')

In [24]:
def train_one_epoch(model, optimizer, dataloader):
    model.train()
    running_loss = 0
    runs = 0

    for inputs, alpha_labels,_,_ in dataloader:

        if runs >= 10000:
            break

        inputs, alpha_labels = inputs.to(DEVICE), alpha_labels.to(DEVICE)
        mask = (alpha_labels != LABEL_PADDING_VALUE).float()

        outputs = model(inputs)
        outputs = outputs.squeeze(-1)
        total_loss = (continuous_loss_fn(outputs, alpha_labels) * mask).sum() / mask.sum()
                
        optimizer.zero_grad()
        total_loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()
        
        running_loss += total_loss.item()
        runs += 1

    return running_loss/runs


def evaluate_model(model, dataloader):
    model.eval()
    
    running_val_total = 0.0
    val_runs = 0

    with torch.no_grad():
        for inputs, alpha_labels,_,_ in dataloader:

            if val_runs >= 10000:
                break
            
            inputs, alpha_labels = inputs.to(DEVICE), alpha_labels.to(DEVICE)
            mask = (alpha_labels != LABEL_PADDING_VALUE).float()
            
            outputs = model(inputs)  
            outputs = outputs.squeeze(-1)
            loss_alpha = (continuous_loss_fn(outputs, alpha_labels) * mask).sum() / mask.sum()            
            running_val_total += loss_alpha.item()
            val_runs += 1
    
    return running_val_total / val_runs

# Objective Function

Here you can change the code to tune either model architecture, learning rates, epochs, batch_size, etc.

In [27]:
def objective(trial):
    # Hyperparameter suggestions
    l2_lambda = trial.suggest_float("lambda_l2", 1e-6, 1e-1, log=True)
    learning_rate = trial.suggest_float("lr", 1e-6, 1e-1, log=True)

    # Initialize model and optimizer
    model = Seq2Seq(num_inputs=NUM_FEATURES).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=l2_lambda)
    best_val_loss = float("inf")

    for epoch in range(3): 
        
        train_one_epoch(model, optimizer=optimizer, dataloader=training_loader)
        val_total_loss = evaluate_model(model, val_loader)
        
        # Report intermediate loss to Optuna
        trial.report(val_total_loss, epoch)

        # Prune trial if it should be pruned
        # if trial.should_prune():
        #     raise optuna.TrialPruned()

        # Update best validation loss
        if val_total_loss < best_val_loss:
            best_val_loss = val_total_loss

    return best_val_loss

In [ ]:
os.makedirs("optuna_study", exist_ok=True)
storage_path = "sqlite:///optuna_study/tune_alpha.db"

# pruner = optuna.pruners.MedianPruner()
study = optuna.create_study(direction="minimize",
                            # pruner=pruner,
                            storage=storage_path)

study.optimize(objective, n_trials=100)

# Plot parameters

In [ ]:
# # Access the study using the storage path
storage_path = "sqlite:///optuna_study/runs.db"
study = optuna.load_study(study_name='no-name-cf419b45-bc96-4877-9ae5-25fff167dfdb', storage=storage_path)

# Plot optimization history
fig1 = vis.plot_optimization_history(study, target=lambda t: t.values[0], target_name="Alpha Loss")
fig1.show()

# Plot hyperparameter importances
fig2 = vis.plot_param_importances(study)
fig2.show()

# Plot hyperparameter relationships (example for lambda_l2 vs objective value)
fig3 = vis.plot_slice(study, params=['lambda_l2'], target=lambda t: t.values[0], target_name="Total Loss")
fig3.show()

fig4 = vis.plot_slice(study, params=['lr'], target=lambda t: t.values[0], target_name="Total Loss")
fig4.show()
